# COSMIC PREDICT — JEPA-inspired training

This notebook trains the compact ViT-style JEPA baseline on `ObservationPatchDataset` and writes a checkpoint for the API. The current downloaded FITS set does not yet include a verified same-footprint temporal pair manifest, so the default run is an **encoder smoke/pretraining run** using each real patch as both context and target. It is not evidence of temporal anomaly performance. Replace the target construction with `ObservationPairDataset` after WCS footprint, band, and epoch compatibility have been verified.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "ai" and ROOT.parent.name == "backend":
    ROOT = ROOT.parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

from backend.ai.dataset import ObservationPatchDataset
from backend.ai.predictor import JEPARepresentationModel

DATASET_MANIFEST = ROOT / "data" / "processed" / "manifest.json"
CHECKPOINT_PATH = ROOT / "models" / "checkpoints" / "jepa.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
print({"device": str(DEVICE), "dataset": str(DATASET_MANIFEST)})


{'device': 'cuda', 'dataset': 'D:\\NasaSpaceBalls\\data\\processed\\manifest.json'}


In [2]:
if not DATASET_MANIFEST.is_file():
    raise FileNotFoundError(
        f"{DATASET_MANIFEST} is missing; run scripts/preprocess.py first."
    )

dataset = ObservationPatchDataset(DATASET_MANIFEST)
sample = dataset[0]
print("samples:", len(dataset))
print("image:", tuple(sample["image"].shape), "valid pixels:", int(sample["valid_mask"].sum()))


samples: 768
image: (1, 128, 128) valid pixels: 3829


The model predicts a latent grid. Pixel masks are downsampled to that grid so invalid FITS pixels do not contribute to the loss. For a real JEPA temporal run, construct context/target batches from an explicit verified pair manifest instead of reusing the same patch.

In [3]:
train_size = max(1, int(len(dataset) * 0.8))
validation_size = len(dataset) - train_size
split_generator = torch.Generator().manual_seed(SEED)
train_set, validation_set = random_split(
    dataset, [train_size, validation_size], generator=split_generator
)
train_loader = DataLoader(train_set, batch_size=8, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_set, batch_size=8, shuffle=False, num_workers=0) if validation_size else None

model = JEPARepresentationModel(
    in_channels=1,
    latent_dim=64,
    base_channels=32,
    predictor_hidden_dim=128,
).to(DEVICE)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=1e-4,
)

MIN_VALID_FRACTION = 0.25

def latent_valid_mask(pixel_mask):
    # A strict all-pixels rule discards every latent patch in these FITS files.
    # Keep a latent location when at least 25% of its source pixels are valid.
    return F.avg_pool2d(
        pixel_mask.float().unsqueeze(1), kernel_size=8, stride=8
    ).squeeze(1) >= MIN_VALID_FRACTION


In [4]:
def run_epoch(loader, training):
    model.train(training)
    losses = []
    for batch in loader:
        images = batch["image"].to(DEVICE)
        valid_mask = latent_valid_mask(batch["valid_mask"].to(DEVICE))
        if not valid_mask.any():
            continue
        # Bootstrap target: replace with a verified later observation for JEPA training.
        context, target = images, images
        with torch.set_grad_enabled(training):
            loss = model.loss(context, target, valid_mask)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                model.update_target_encoder()
        losses.append(float(loss.detach().cpu()))
    if not losses:
        raise RuntimeError(
            "no usable latent locations in this epoch; lower MIN_VALID_FRACTION "
            "or inspect the FITS quality masks"
        )
    return sum(losses) / len(losses)

history = []
for epoch in range(5):
    train_loss = run_epoch(train_loader, training=True)
    validation_loss = run_epoch(validation_loader, training=False) if validation_loader else None
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "validation_loss": validation_loss})
    print(history[-1])


{'epoch': 1, 'train_loss': 0.25294353551678844, 'validation_loss': 0.1469167023897171}
{'epoch': 2, 'train_loss': 0.10100756290477592, 'validation_loss': 0.0687410619109869}
{'epoch': 3, 'train_loss': 0.05480960843625007, 'validation_loss': 0.048447967320680616}
{'epoch': 4, 'train_loss': 0.043075475599858665, 'validation_loss': 0.040765249729156496}
{'epoch': 5, 'train_loss': 0.03572620850588594, 'validation_loss': 0.032193691655993464}


In [5]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "architecture": {
            "in_channels": 1,
            "latent_dim": 64,
            "base_channels": 32,
            "predictor_hidden_dim": 128,
            "patch_size": 8,
        },
        "seed": SEED,
        "history": history,
        "training_note": "Bootstrap same-patch target; not temporal evaluation.",
    },
    CHECKPOINT_PATH,
)
print(f"saved checkpoint: {CHECKPOINT_PATH}")


saved checkpoint: D:\NasaSpaceBalls\models\checkpoints\jepa.pt


In [6]:
from backend.ai.inference import JEPAInference

inference = JEPAInference.from_checkpoint(CHECKPOINT_PATH)
result = inference.run(sample["image"].numpy(), sample["image"].numpy())
print({"status": result.status, "jepa_score": result.jepa_score, "score_kind": result.score_kind})


{'status': 'ok', 'jepa_score': 0.03506504371762276, 'score_kind': 'raw_latent_mean_squared_error'}
